In [1]:
import pandas as pd
import xarray as xr
import xcdat as xc
import numpy as np
import os
import glob
import re
import cftime
import calendar

/global/homes/j/jungchoi/.conda/envs/pcmdi_metrics/lib/python3.10/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


Model information, Target lead time

In [2]:
print("%%%%%%%% start %%%%%%%")
output_dir = "/pscratch/sd/j/jungchoi/DCPP/_metrics"
output_grid = xc.regridder.grid.create_uniform_grid(-88.75, 88.75, 2.5, 0.0, 357.5, 2.5)
output_grid_no = "144x72"

input_dir = "/pscratch/sd/j/jungchoi/DCPP/_link"
experiment = "dcppA-hindcast"

trend_year_start = 1979
trend_year_end = 2014

trend_years = list(range(trend_year_start, trend_year_end + 1))
print(trend_years)

%%%%%%%% start %%%%%%%
[1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014]


# Target leadtime average, Horizonal interpolarion for each ensemble members and initializations

In [ ]:
# Loop over ensemble members and initialization years
#model = "CanESM5"
#model = "CMCC-CM2-SR5"
#model = "HadGEM3-GC31-MM"
#model = "EC-Earth3"
#model = "CNRM-ESM2-1"
#model = "FGOALS-f3-L"
#model = "IPSL-CM6A-LR"
#model = "MIROC6"
#model = "MPI-ESM1-2-HR"
#model = "MRI-ESM2-0"
#model = "NorCPM1"

#mdl_var_name = "tas"
mdl_var_name = "pr"

model_list = ['CanESM5', 'CMCC-CM2-SR5', 'CNRM-ESM2-1', 'EC-Earth3', 'FGOALS-f3-L', 'HadGEM3-GC31-MM', 'IPSL-CM6A-LR', 'MIROC6', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NorCPM1']
#model_list = ['CanESM5']
#model_list = ['HadGEM3-GC31-MM', 'IPSL-CM6A-LR', 'MIROC6', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NorCPM1']

for model in model_list:

    lead_year_end = 10
    if model in ["CNRM-ESM2-1", "MRI-ESM2-0"]:
        lead_year_end = 5
    
    ensemble_no = 10
    if( model == "FGOALS-f3-L" ):
        ensemble_no = 9
    
    for lead_year in range(1, lead_year_end + 1):
        ensemble_ds = []
    
        target_initialization_start = trend_year_start - lead_year
        target_initialization_end = trend_year_end - lead_year
        
        for ensemble in range(1, ensemble_no + 1):
            each_init_ds = []
            
            for init_year in range(target_initialization_start, target_initialization_end + 1):
                exp_name = f"s{init_year}"        
                target_year = init_year + lead_year
            
                file_pattern = os.path.join(input_dir, model, exp_name, f"{mdl_var_name}_r{ensemble}_*.nc")
                file_list = sorted(glob.glob(file_pattern))  # Get all matching files
    
                #print(file_list)
                if not file_list:
                    print(f"%%% Error: No files found in: {input_dir}/{model}/{exp_name}")
                    continue  # Skip if no files found
                
                nfile = len(file_list)            
                selected_file = []
                #print(file_list, nfile)
    
                if nfile == 1: 
                    selected_file.append(file_list[0])
                    print(init_year, target_year, selected_file)
                    
                    ds = xr.open_dataset(selected_file[0])
                    time_values = ds["time"].values
                    selected_ds = ds.sel(time=ds.time.dt.year == target_year)
                    regrid_ds = selected_ds.regridder.horizontal(f"{mdl_var_name}", output_grid, tool="regrid2")
                    ds.close()
            
                if nfile > 1:
                    for file in file_list:
                        period = file.split("_")[-1]
                        start_str, end_str = period.split("-")
        
                        start_year = int(start_str[:4])
                        end_year = int(end_str[:4])
        
                        if int(target_year) >= start_year and int(target_year) <= end_year:
                            selected_file.append(file)
                    print(init_year, target_year, selected_file)
    
                    filtered_ds = []
                    for file in selected_file:       
                        ds = xr.open_dataset(file)
                        time_values = ds["time"].values
                        selected_ds = ds.sel(time=ds.time.dt.year == target_year)
                        filtered_ds.append(selected_ds[f"{mdl_var_name}"])
                    merged_ds = xr.concat(filtered_ds, dim="time")   
                    merged_ds = merged_ds.to_dataset(name=f"{mdl_var_name}")    
                    regrid_ds = merged_ds.regridder.horizontal(f"{mdl_var_name}", output_grid, tool="regrid2")
                    ds.close()
       
                each_init_ds.append(regrid_ds)    
            
            all_init_ds = xr.concat(each_init_ds, dim="time")
            ensemble_ds.append(all_init_ds)
        #    print(all_init_ds.time)
    
        all_ensemble_ds = xr.concat(ensemble_ds, dim="ensemble")
        #print('%% lead==', all_ensemble_ds)
        #print(all_ensemble_ds.time)
    
        emean_ds = all_ensemble_ds.mean(dim="ensemble")
        da = emean_ds[f"{mdl_var_name}"]

        if mdl_var_name == "pr":
            da = da * 60 * 60 * 24

        years = emean_ds.time.dt.year
        all_years = years.values.astype(float) 

        monthly_slopes = []

        for month in range(1, 13):
            da_month = da.where(emean_ds.time.dt.month == month, drop=True)
            x = years.where(emean_ds.time.dt.month == month, drop=True).values.astype(float)

            def fit_slope(x, y):
                if np.isfinite(y).sum() < 10:
                    return np.nan
                return np.polyfit(x, y, deg=1)[0]

            trend = xr.apply_ufunc(
                fit_slope,
                xr.DataArray(x, dims="time"),  # x
                da_month,                      # y
                input_core_dims=[["time"], ["time"]],
                output_core_dims=[[]],
                vectorize=True,
                dask="parallelized",
                output_dtypes=[float],
            )

            monthly_slopes.append(trend)

        monthly_trend = xr.concat(monthly_slopes, dim="month")
        monthly_trend["month"] = np.arange(1, 13)
        monthly_trend.name = f"{mdl_var_name}"
        monthly_trend = monthly_trend.to_dataset()
        print(monthly_trend)
        
        # Save to NetCDF for each lead time
        output_filename = f"{output_dir}/{model}/{mdl_var_name}.{output_grid_no}.YR{lead_year}.mon_trend{trend_year_start}-{trend_year_end}.em.nc"
        if os.path.exists(output_filename):
            os.remove(output_filename)
        monthly_trend.to_netcdf(output_filename)
        print(f"%% Regridded model monthly trend saved: {output_filename}")

1978 1979 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1978/pr_r1_197901-198812.nc']
1979 1980 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1979/pr_r1_198001-198912.nc']
1980 1981 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1980/pr_r1_198101-199012.nc']
1981 1982 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1981/pr_r1_198201-199112.nc']
1982 1983 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1982/pr_r1_198301-199212.nc']
1983 1984 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1983/pr_r1_198401-199312.nc']
1984 1985 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1984/pr_r1_198501-199412.nc']
1985 1986 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1985/pr_r1_198601-199512.nc']
1986 1987 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1986/pr_r1_198701-199612.nc']
1987 1988 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1987/pr_r1_198801-199712.nc']
1988 1989 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5/s1988/pr_r1_198901-199812.nc']
1989 1990 ['/pscratch/sd/j/jungchoi/DCPP/_link/CanESM5

In [4]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


time_str = [str(t) for t in all_ensemble_ds.time.values]
time_str_ym = [ts[:7] for ts in time_str]
#time_array = all_ensemble_ds.time.values
#conv_time = pd.to_datetime(time_array)
#mpl_dates = mdates.date2num(time_array)

n_ens = all_ensemble_ds.sizes["ensemble"]  # 또는 n_ens = all_ensemble_ds.ensemble.size

plt.figure(figsize=(12, 6))
#print(time)
#conv_time = [t.datetime for t in time_list]

#for ens in range(0, 1):
tas_series = all_ensemble_ds.tas[0, :, 36, 72].values
plt.plot(time_str_ym, tas_series, label=f"Ens")

x_pos = np.arange(len(time_str_ym))
step = 12
plt.xticks(ticks=x_pos[::step], labels=np.array(time_str_ym)[::step], rotation=90)

plt.title("Temperature Time Series at (lat=36, lon=72) - All Ensembles")
plt.xlabel("Time")
plt.ylabel("Temperature (tas)")
#plt.legend(ncol=2, fontsize="small")
plt.grid(True)
plt.tight_layout()
plt.show()


 

AttributeError: 'Dataset' object has no attribute 'tas'

<Figure size 1200x600 with 0 Axes>